# Set up

In [ ]:
# !pip install datasets transformers pandas matplotlib tqdm --upgrade --quiet

In [85]:
import datasets
from transformers import pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import re
import json
import os
import time
import openai
from sklearn.metrics import cohen_kappa_score, classification_report

In [86]:
import sys
sys.path.append('../')
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

sentences = ["I am not having a great day"]

model_outputs = classifier(sentences)
print(model_outputs[0])
# produces a list of dicts for each of the labels


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

[{'label': 'disappointment', 'score': 0.46669575572013855}, {'label': 'sadness', 'score': 0.3984948694705963}, {'label': 'annoyance', 'score': 0.0680660679936409}, {'label': 'neutral', 'score': 0.05703020095825195}, {'label': 'disapproval', 'score': 0.04423939809203148}, {'label': 'nervousness', 'score': 0.014850758947432041}, {'label': 'realization', 'score': 0.014059898443520069}, {'label': 'approval', 'score': 0.011267454363405704}, {'label': 'joy', 'score': 0.006303384434431791}, {'label': 'remorse', 'score': 0.006221492774784565}, {'label': 'caring', 'score': 0.006029391661286354}, {'label': 'embarrassment', 'score': 0.005265495739877224}, {'label': 'anger', 'score': 0.0049814400263130665}, {'label': 'disgust', 'score': 0.004259037785232067}, {'label': 'grief', 'score': 0.004002136643975973}, {'label': 'confusion', 'score': 0.003382926108315587}, {'label': 'relief', 'score': 0.0031404944602400064}, {'label': 'desire', 'score': 0.0028274687938392162}, {'label': 'admiration', 'score

In [50]:
def get_goemo_score(output):
    return {i['label']: i['score'] for i in output if i['label'] in ['anger', 'disgust', 'fear', 'sadness']}
get_goemo_score(model_outputs[0])

{'sadness': 0.5617969632148743,
 'anger': 0.004118029028177261,
 'disgust': 0.003437246195971966,
 'fear': 0.0019833652768284082}

In [51]:
def get_goemo_label(output):
    # in the repo they used threshold 0.5 for the benchmark (since the label is binary)
    return {i['label']: 0 if i['score'] < 0.5 else 1 for i in output if i['label'] in ['anger', 'disgust', 'fear', 'sadness']}
get_goemo_label(model_outputs[0])

{'sadness': 1, 'anger': 0, 'disgust': 0, 'fear': 0}

In [54]:
sentences = ["I’m really sorry about your situation :(", "I'm relieved that it's finally over.", "I'm a bit worried about our situation especially when we live together."]
model_outputs = classifier(sentences)
[get_goemo_label(output) for output in model_outputs]

[{'sadness': 1, 'anger': 0, 'disgust': 0, 'fear': 0},
 {'sadness': 0, 'fear': 0, 'anger': 0, 'disgust': 0},
 {'fear': 0, 'sadness': 0, 'anger': 0, 'disgust': 0}]

# Extension: NRCLex

In [37]:
!pip install nrclex

In [23]:
!python -m textblob.download_corpora

[nltk_data] Downloading package brown to /Users/ngocdoan/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ngocdoan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ngocdoan/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/ngocdoan/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to
[nltk_data]     /Users/ngocdoan/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/ngocdoan/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.


In [45]:
from nrclex import NRCLex

text_object = NRCLex()
text_object.load_raw_text("I'm a bit worried about our situation especially when we live together.")


In [46]:
text_object.raw_emotion_scores

{'negative': 1, 'sadness': 1}

In [47]:
test = text_object.raw_emotion_scores

for emo in ['anger', 'fear', 'disgust', 'sadness']:
    if emo in test: print({emo: 1})
    else: print({emo: 0})

{'anger': 0}
{'fear': 0}
{'disgust': 0}
{'sadness': 1}


In [49]:
result = text_object.raw_emotion_scores

def get_nrc_label(nrc_res):
    return {emo: 1 if emo in nrc_res else 0 for emo in ['anger', 'fear', 'disgust', 'sadness']}
print(get_nrc_label(result))

{'anger': 0, 'fear': 0, 'disgust': 0, 'sadness': 1}


# Emotion detection on 100-post sample

Here we try running NRCLex (lexicon approach) and GoEmotions-finetuned RoBERTa model (off-the-shelf model) on emotion detection task and compare that to GPT on our human annotation set (via Kappa's)

In [ ]:
sample_annot = pd.read_csv('sample_emo_annot.csv')

(Rewrite label functions:)

In [81]:
def get_nrc_label(text: str) -> dict:
    obj = NRCLex()
    obj.load_raw_text(text)
    scores = obj.raw_emotion_scores
    return {emo: 1 if emo in scores else 0
            for emo in ['anger', 'fear', 'sadness', 'disgust']}

def get_goemo_label(text: str, threshold=0.5) -> dict:
    output = classifier([text])[0]
    return {i['label']: 0 if i['score'] < threshold else 1
            for i in output
            if i['label'] in ['anger', 'disgust', 'fear', 'sadness']}

Annotate the dataframe

In [82]:
def annotate_df(df: pd.DataFrame) -> pd.DataFrame:
    nrc_records, goemo_records = [], []

    for text in df['selftext']:
        nrc   = get_nrc_label(text)
        goemo = get_goemo_label(text, threshold=0.3)

        nrc_records.append({
            'anger (NRC)':   nrc.get('anger',   0),
            'fear (NRC)':    nrc.get('fear',     0),
            'sad (NRC)':     nrc.get('sadness',  0),   # NRCLex uses 'sadness'
            'disgust (NRC)': nrc.get('disgust',  0),
        })
        goemo_records.append({
            'anger (GoEmo)':   goemo.get('anger',   0),
            'fear (GoEmo)':    goemo.get('fear',     0),
            'sad (GoEmo)':     goemo.get('sadness',  0),  # GoEmotions uses 'sadness'
            'disgust (GoEmo)': goemo.get('disgust',  0),
        })

    df = df.copy()
    df = pd.concat([df,
                    pd.DataFrame(nrc_records,   index=df.index),
                    pd.DataFrame(goemo_records, index=df.index)], axis=1)
    return df

sample_annot = annotate_df(sample_annot)

Kappa's calculation and comparison

In [ ]:
EMOTIONS = ['anger', 'fear', 'sad', 'disgust']

# Map each method to its column suffix and the emotion key used above
METHODS = {
    'GPT':   '(L)',
    'NRC':   '(NRC)',
    'GoEmo': '(GoEmo)',
}
HUMAN_REFS = ['(R)', '(G)']

def kappa_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method, suffix in METHODS.items():
        for ref_suffix in HUMAN_REFS:
            ref_label = 'Human-R' if ref_suffix == '(R)' else 'Human-G'
            kappas = {}
            for emo in EMOTIONS:
                pred = df[f'{emo} {suffix}']
                true = df[f'{emo} {ref_suffix}']
                kappas[emo] = round(cohen_kappa_score(true, pred), 3)
            kappas['mean'] = round(sum(kappas.values()) / len(EMOTIONS), 3)
            rows.append({'method': method, 'vs': ref_label, **kappas})

    return pd.DataFrame(rows).set_index(['method', 'vs'])

kappa_df = kappa_table(sample_annot)
print(kappa_df.to_string())

                anger   fear    sad  disgust   mean
method vs                                          
GPT    Human-R  0.583  0.779  0.658    0.820  0.710
       Human-G  0.464  0.702  0.260    0.575  0.500
NRC    Human-R -0.020  0.000 -0.016    0.043  0.002
       Human-G  0.019  0.000  0.138    0.059  0.054
GoEmo  Human-R  0.000  0.077  0.025    0.000  0.026
       Human-G  0.000  0.090  0.078    0.000  0.042


# Benchmark GPT Annotation on GoEmotions

Tbh the whole point is that we use GPT because the context of our text (radicalization narratives) is niche/contextual so that models trained on generic text is not suitable, and we're benchmarking on generic text dataset...?? But let's just try anw ig

Or maybe the point is GPT performs well with generic text so it should also (at least) do well in more contextual, rich text?

In [ ]:
# Test run GPT
openai.api_key = os.environ.get("DLAB_OPENAI_KEY")

response = openai.ChatCompletion.create(
    model="gpt-4o-mini-2024-07-18",
    messages=[{"role": "user", "content": "Hello how are u"}]
)

In [65]:
response.choices[0].message.content.strip()

"Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"

In [ ]:
# Our old code for emotion detection on a post (just for reference)
basic_emotions = [
    'anger', 'fear', 'sadness', 'disgust'
]

def get_llm_emotion_detection(text):
    prompt = f"""
You are an expert in emotion analysis, and you are analyzing posts written by people in the subreddit r/QAnonCasualties.
Each post is a personal narrative written by a narrator (the poster) describing their experiences with a loved one who has been radicalized.
Your task is to determine whether the narrator expresses each of the following emotions in the post: Anger, Disgust, Sadness, Fear, and Surprise.
We are only interested in the narrator’s emotions, not the emotions of other people mentioned in the post.
For each emotion, output 1 if the narrator expresses it at least once, otherwise output 0.
We are not scoring how strong the emotion is, only whether it is present.

Text: "{text}"

Please provide outputs for these emotions in JSON format:
{{
    "anger": 0 or 1,
    "fear": 0 or 1,
    "sadness": 0 or 1,
    "disgust": 0 or 1,
    "surprise": 0 or 1,
}}

Only respond with the JSON object, no additional text.
"""
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4o-mini-2024-07-18",
            temperature=0.2,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.choices[0].message.content.strip()
        answer = re.sub(r'^```(?:json)?|```$', '', answer.strip(), flags=re.MULTILINE).strip()
        return json.loads(answer)
        
    except Exception as e:
        print(f"Error: {e}")
        return {emotion: 0.0 for emotion in basic_emotions}

In [ ]:
# GoEmotions test set
split_name = "test"

dataset_name, dataset_config_name = "go_emotions", "simplified"
dataset_dict = datasets.load_dataset(dataset_name, dataset_config_name)
dataset_dict[split_name][0]

README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'text': 'I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!',
 'labels': [25],
 'id': 'eecwqtt'}

In [84]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [69]:
labels = dataset_dict[split_name].features["labels"].feature.names
print({i: l for i, l in enumerate(labels)})

{0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance', 4: 'approval', 5: 'caring', 6: 'confusion', 7: 'curiosity', 8: 'desire', 9: 'disappointment', 10: 'disapproval', 11: 'disgust', 12: 'embarrassment', 13: 'excitement', 14: 'fear', 15: 'gratitude', 16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness', 20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief', 24: 'remorse', 25: 'sadness', 26: 'surprise', 27: 'neutral'}


In [ ]:
# again we only care about anger, disgust, fear and sadness, so let's just benchmark on that
# but again it's kinda weird because we use different prompt also so it doesn't directly connect to our implementation??

Load & filter GoEmotions test set

In [87]:
TARGET_EMOTIONS = {'anger': 2, 'fear': 14, 'sadness': 25, 'disgust': 11}
# ^ emotion name → GoEmotions label index

dataset_dict = datasets.load_dataset("go_emotions", "simplified")
test_set = dataset_dict["test"]
labels_list = test_set.features["labels"].feature.names

def has_target_emotion(example):
    """Keep only examples that contain at least one of the 4 target emotions."""
    return any(idx in example["labels"] for idx in TARGET_EMOTIONS.values())

filtered = test_set.filter(has_target_emotion)
print(f"Filtered test set: {len(filtered)} examples (from {len(test_set)})")

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

Filtered test set: 542 examples (from 5427)


In [88]:
# Build ground-truth binary vectors
def make_gt_vector(example):
    return {emo: int(idx in example["labels"])
            for emo, idx in TARGET_EMOTIONS.items()}

# GPT annotation
openai.api_key = os.environ.get("DLAB_OPENAI_KEY")

SYSTEM_PROMPT = """You are an emotion classification assistant.
Given a text, identify which of the following emotions are expressed (there may be zero, one, or multiple):
- anger
- fear
- sadness
- disgust

Respond ONLY with a JSON object with these exact keys and binary values (0 or 1). Example:
{"anger": 0, "fear": 1, "sadness": 1, "disgust": 0}
No explanation, no extra text."""

def get_gpt_label(text: str, retries: int = 3, wait: float = 2.0) -> dict:
    for attempt in range(retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4o-mini-2024-07-18",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": text}
                ],
                temperature=0,          # deterministic
                max_tokens=50,
            )
            raw = response.choices[0].message.content.strip()
            parsed = json.loads(raw)
            # Normalise: ensure all 4 keys exist and values are 0/1
            return {emo: int(bool(parsed.get(emo, 0)))
                    for emo in TARGET_EMOTIONS}
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  Parse error on attempt {attempt+1}: {e} | raw='{raw}'")
        except openai.error.RateLimitError:
            print(f"  Rate limit hit, waiting {wait}s...")
            time.sleep(wait)
            wait *= 2   # exponential back-off
    # Fallback: all zeros if all retries fail
    return {emo: 0 for emo in TARGET_EMOTIONS}

In [89]:
# Run annotation (with checkpoint saving) 
CHECKPOINT_PATH = "gpt_goemo_checkpoint.jsonl"

# Resume from checkpoint if it exists
annotated_ids = set()
results = []
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            record = json.loads(line)
            results.append(record)
            annotated_ids.add(record["id"])
    print(f"Resumed from checkpoint: {len(results)} already annotated")

with open(CHECKPOINT_PATH, "a") as checkpoint_file:
    for i, example in enumerate(filtered):
        if example["id"] in annotated_ids:
            continue

        gpt_pred = get_gpt_label(example["text"])
        gt        = make_gt_vector(example)

        record = {
            "id":   example["id"],
            "text": example["text"],
            **{f"gt_{emo}":  gt[emo]       for emo in TARGET_EMOTIONS},
            **{f"gpt_{emo}": gpt_pred[emo] for emo in TARGET_EMOTIONS},
        }
        results.append(record)
        checkpoint_file.write(json.dumps(record) + "\n")
        checkpoint_file.flush()

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(filtered)}] annotated...")
        
        time.sleep(0.1)   # gentle rate limiting

  [50/542] annotated...
  [100/542] annotated...
  [150/542] annotated...
  [200/542] annotated...
  [250/542] annotated...
  [300/542] annotated...
  [350/542] annotated...
  [400/542] annotated...
  [450/542] annotated...
  [500/542] annotated...


In [ ]:
# Evaluation
df_eval = pd.DataFrame(results)

print("\n" + "="*60)
print("CLASSIFICATION REPORT (per emotion, GPT vs ground truth)")
print("="*60)
for emo in TARGET_EMOTIONS:
    print(f"\n── {emo.upper()} ──")
    print(classification_report(
        df_eval[f"gt_{emo}"], df_eval[f"gpt_{emo}"],
        target_names=[f"no {emo}", emo], zero_division=0
    ))


CLASSIFICATION REPORT (per emotion, GPT vs ground truth)

── ANGER ──
              precision    recall  f1-score   support

    no anger       0.92      0.74      0.82       344
       anger       0.67      0.88      0.76       198

    accuracy                           0.80       542
   macro avg       0.79      0.81      0.79       542
weighted avg       0.83      0.80      0.80       542


── FEAR ──
              precision    recall  f1-score   support

     no fear       0.96      0.92      0.94       464
        fear       0.63      0.77      0.69        78

    accuracy                           0.90       542
   macro avg       0.80      0.85      0.82       542
weighted avg       0.91      0.90      0.91       542


── SADNESS ──
              precision    recall  f1-score   support

  no sadness       0.92      0.86      0.89       386
     sadness       0.70      0.81      0.75       156

    accuracy                           0.85       542
   macro avg       0.81      0